In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install torch torchvision segmentation-models-pytorch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.8 MB/s eta 0:00:00


In [3]:
import torch
import numpy as np
import os

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import torchvision.transforms.functional as TF

from sklearn.metrics import classification_report



In [4]:
PROJECT = '/content/drive/MyDrive/GeoVLM_Project'
DATA_PROCESSED = f'{PROJECT}/data/processed'
MODELS = f'{PROJECT}/models'

os.makedirs(MODELS, exist_ok=True)


#  GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using: {device}")

Using: cuda


In [5]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


Load the Processed data


In [6]:
X_train = np.load(f'{DATA_PROCESSED}/X_train.npy')
X_val   = np.load(f'{DATA_PROCESSED}/X_val.npy')
y_train = np.load(f'{DATA_PROCESSED}/y_train.npy')
y_val   = np.load(f'{DATA_PROCESSED}/y_val.npy')

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_val:   {y_val.shape}")

X_train: (1928, 256, 256, 6)
X_val:   (483, 256, 256, 6)
y_train: (1928, 256, 256)
y_val:   (483, 256, 256)


In [7]:
import random

class DisasterDataset(Dataset):
    def __init__(self, images, masks, augment=False):
        self.images  = torch.tensor(images).permute(0, 3, 1, 2)
        self.masks   = torch.tensor(masks).long()
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        mask  = self.masks[idx]

        if self.augment:
            # Random horizontal flip
            if random.random() > 0.5:
                image = TF.hflip(image)
                mask  = TF.hflip(mask.unsqueeze(0)).squeeze(0)

            # Random vertical flip
            if random.random() > 0.5:
                image = TF.vflip(image)
                mask  = TF.vflip(mask.unsqueeze(0)).squeeze(0)

            # Random rotation
            if random.random() > 0.5:
                angle = random.choice([90, 180, 270])
                image = TF.rotate(image, angle)
                mask  = TF.rotate(mask.unsqueeze(0), angle).squeeze(0)

        return image, mask

# Augmentation ON for training, OFF for validation
train_dataset = DisasterDataset(X_train, y_train, augment=True)
val_dataset   = DisasterDataset(X_val,   y_val,   augment=False)

train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=0, pin_memory=True )

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

Train batches: 241
Val batches:   61


Model Selection

In [8]:
model = smp.Unet(
    encoder_name='resnet50',
    encoder_weights='imagenet',
    in_channels=6,
    classes=5,
    activation=None
).to(device)

print("U-Net ResNet50 built ✓")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

U-Net ResNet50 built ✓
Parameters: 32,531,093


Loss Funtions : BCE + Dice Loss


In [9]:
class_weights = torch.tensor([
    0.1,   # background
    0.5,   # no damage
    1.5,   # minor damage
    2.0,   # major damage
    3.0    # destroyed
]).to(device)

class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce  = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = smp.losses.DiceLoss(
            mode='multiclass',
            classes=5,
            smooth=1.0
        )

    def forward(self, pred, target):
        bce_loss  = self.bce(pred, target)
        dice_loss = self.dice(pred, target)
        return 0.5 * bce_loss + 0.5 * dice_loss

criterion = BCEDiceLoss()
print("BCE + Dice loss ready ✓")

BCE + Dice loss ready ✓


 Optimizer and scheduler

In [10]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience=3,
    factor=0.5,
    # verbose=True
)

print("Optimizer ready ✓")


Optimizer ready ✓


In [11]:
import matplotlib.pyplot as plt
import os

os.makedirs(f'{PROJECT}/outputs', exist_ok=True)

EPOCHS = 30
best_val_loss = float('inf')
patience_counter = 0
patience_limit = 7

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):

    # ───────── TRAIN ─────────
    model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, masks in train_loader:
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item()

        preds = outputs.argmax(dim=1)
        train_correct += (preds == masks).sum().item()
        train_total += masks.numel()

    train_loss = train_loss_sum / len(train_loader)
    train_acc = 100 * train_correct / train_total


    # ───────── VALIDATION ─────────
    model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device, non_blocking=True)
            masks  = masks.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, masks)

            val_loss_sum += loss.item()

            preds = outputs.argmax(dim=1)
            val_correct += (preds == masks).sum().item()
            val_total += masks.numel()

    val_loss = val_loss_sum / len(val_loader)
    val_acc = 100 * val_correct / val_total


    scheduler.step(val_loss)


    # ───────── STORE METRICS ─────────
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)


    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss {train_loss:.4f} | Val Loss {val_loss:.4f} | "
        f"Train Acc {train_acc:.2f}% | Val Acc {val_acc:.2f}%"
    )


    # ───────── SAVE BEST MODEL ─────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), f'{MODELS}/best_model_resnet50.pth')
        print("✓ Best model saved")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience_limit})")


    # ───────── PLOT (ONLY EVERY 5 EPOCHS) ─────────
    if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        axes[0].plot(train_losses, label='Train Loss')
        axes[0].plot(val_losses, label='Val Loss')
        axes[0].set_title("Loss Curve")
        axes[0].legend()

        axes[1].plot(train_accs, label='Train Acc')
        axes[1].plot(val_accs, label='Val Acc')
        axes[1].set_title("Accuracy Curve")
        axes[1].legend()

        plt.tight_layout()
        plt.savefig(f'{PROJECT}/outputs/training_curves.png')
        plt.close(fig)


    # ───────── EARLY STOPPING ─────────
    if patience_counter >= patience_limit:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break


print("\nTraining complete ✓")
print(f"Best val loss: {best_val_loss:.4f}")

Epoch 1/30 | Train Loss 0.8497 | Val Loss 1.8909 | Train Acc 79.90% | Val Acc 53.38%
✓ Best model saved
Epoch 2/30 | Train Loss 0.7713 | Val Loss 0.7456 | Train Acc 83.79% | Val Acc 88.57%
✓ Best model saved
Epoch 3/30 | Train Loss 0.7251 | Val Loss 0.7110 | Train Acc 82.95% | Val Acc 79.76%
✓ Best model saved
Epoch 4/30 | Train Loss 0.7069 | Val Loss 0.8100 | Train Acc 83.53% | Val Acc 80.96%
No improvement (1/7)
Epoch 5/30 | Train Loss 0.6878 | Val Loss 0.7694 | Train Acc 84.24% | Val Acc 74.51%
No improvement (2/7)
Epoch 6/30 | Train Loss 0.6630 | Val Loss 1.0593 | Train Acc 84.60% | Val Acc 72.68%
No improvement (3/7)
Epoch 7/30 | Train Loss 0.6520 | Val Loss 0.5937 | Train Acc 84.80% | Val Acc 85.58%
✓ Best model saved
Epoch 8/30 | Train Loss 0.6475 | Val Loss 0.6166 | Train Acc 85.40% | Val Acc 88.35%
No improvement (1/7)
Epoch 9/30 | Train Loss 0.6143 | Val Loss 0.8395 | Train Acc 85.90% | Val Acc 89.46%
No improvement (2/7)
Epoch 10/30 | Train Loss 0.5900 | Val Loss 0.6592 | Tr

In [12]:
# These lists are already in memory from training
print(f"Train losses recorded: {len(train_losses)}")
print(f"Val losses recorded: {len(val_losses)}")

Train losses recorded: 30
Val losses recorded: 30


In [13]:
model.load_state_dict(torch.load(f'{MODELS}/best_model_resnet50.pth'))
model.eval()

all_preds = []
all_masks = []

with torch.no_grad():
    for i, (images, masks) in enumerate(val_loader):
        if i >= 20:
            break
        images  = images.to(device)
        outputs = model(images)
        preds   = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds.flatten()[::10])
        all_masks.extend(masks.numpy().flatten()[::10])

labels = ['background', 'no-damage', 'minor', 'major', 'destroyed']
print(classification_report(all_masks, all_preds, target_names=labels))

              precision    recall  f1-score   support

  background       0.98      0.94      0.96    898591
   no-damage       0.71      0.86      0.78    113348
       minor       0.50      0.70      0.58     18965
       major       0.56      0.89      0.69     14558
   destroyed       0.62      0.89      0.73      3118

    accuracy                           0.93   1048580
   macro avg       0.68      0.86      0.75   1048580
weighted avg       0.94      0.93      0.93   1048580



In [14]:
torch.save({
    'epoch': EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_val_loss': best_val_loss,
}, f'{MODELS}/final_model_resnet50.pth')

print(f"Full model saved ✓")
print(f"Location: {MODELS}/final_model_resnet50.pth")

Full model saved ✓
Location: /content/drive/MyDrive/GeoVLM_Project/models/final_model_resnet50.pth
